In [7]:
import pandas as pd
import glob

# Đọc 1 file thử để xem schema
df_sample = pd.read_csv("../data/raw/MERGED_CSV/Merged01.csv")
print(df_sample.shape)
print(df_sample.columns.tolist())
print(df_sample['Label'].value_counts())

(712311, 40)
['Header_Length', 'Protocol Type', 'Time_To_Live', 'Rate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IGMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Variance', 'Label']
Label
DDOS-ICMP_FLOOD            108662
DDOS-UDP_FLOOD              82011
DDOS-TCP_FLOOD              68289
DDOS-PSHACK_FLOOD           62171
DDOS-RSTFINFLOOD            61652
DDOS-SYN_FLOOD              61460
DDOS-SYNONYMOUSIP_FLOOD     54749
DOS-UDP_FLOOD               50371
DOS-TCP_FLOOD               40391
DOS-SYN_FLOOD               30620
BENIGN                      16577
MIRAI-GREETH_FLOOD          15135
MIRAI-UDPPLAIN              13342
MIRAI-GREIP_FLOOD           11187
DDOS-ICMP_FRAGMENTATION      6784
VULNERABILITYSCAN      

In [11]:
import glob, pandas as pd
from collections import Counter

files = sorted(glob.glob('../data/raw/MERGED_CSV/Merged*.csv'))
print(f"Số file: {len(files)}")  # kỳ vọng 63

label_counts = Counter()
total_rows = 0

for f in files:
    df = pd.read_csv(f, usecols=['Label'])
    label_counts.update(df['Label'].value_counts().to_dict())
    total_rows += len(df)

print(f"Tổng rows: {total_rows:,}")
print(pd.Series(label_counts).sort_values(ascending=False).to_string())

Số file: 63
Tổng rows: 45,019,243
DDOS-ICMP_FLOOD            6893259
DDOS-UDP_FLOOD             5181027
DDOS-TCP_FLOOD             4306086
DDOS-PSHACK_FLOOD          3920372
DDOS-SYN_FLOOD             3886130
DDOS-RSTFINFLOOD           3872808
DDOS-SYNONYMOUSIP_FLOOD    3445659
DOS-UDP_FLOOD              3177323
DOS-TCP_FLOOD              2558256
DOS-SYN_FLOOD              1942176
BENIGN                     1051373
MIRAI-GREETH_FLOOD          949381
MIRAI-UDPPLAIN              852695
MIRAI-GREIP_FLOOD           719655
DDOS-ICMP_FRAGMENTATION     433157
VULNERABILITYSCAN           357583
MITM-ARPSPOOFING            294469
DDOS-UDP_FRAGMENTATION      274909
DDOS-ACK_FRAGMENTATION      272793
DNS_SPOOFING                171468
RECON-HOSTDISCOVERY         128677
RECON-OSSCAN                 93970
RECON-PORTSCAN               78730
DOS-HTTP_FLOOD               68799
DDOS-HTTP_FLOOD              27597
DDOS-SLOWLORIS               22400
DICTIONARYBRUTEFORCE         12522
BROWSERHIJACKING     

In [1]:
import pandas as pd
import numpy as np
import glob, os, pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ── Paths ──────────────────────────────────────────────────────────────
RAW_DIR = '../data/raw/MERGED_CSV/'
OUT_DIR = '../data/processed/'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Load 63 file bằng pandas concat ───────────────────────────────────
files = sorted(glob.glob(RAW_DIR + 'Merged*.csv'))
print(f"Số file: {len(files)}")  # phải ra 63

print("Loading... (~5–10 phút, peak RAM ~14 GB)")
df = pd.concat(
    [pd.read_csv(f) for f in files],
    ignore_index=True
)
print(f"Shape: {df.shape}")  # kỳ vọng (~46.6M, 47)
print(f"Columns: {df.columns.tolist()}")

# # ── Split 80/10/10 stratified ─────────────────────────────────────────
# X = df.drop(columns=['Label'])
# y = df['Label']

# X_tr, X_tmp, y_tr, y_tmp = train_test_split(
#     X, y, test_size=0.2, stratify=y, random_state=42)
# X_val, X_te, y_val, y_te = train_test_split(
#     X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

# print(f"Train: {len(X_tr):,} | Val: {len(X_val):,} | Test: {len(X_te):,}")
# del df, X, y, X_tmp, y_tmp  # giải phóng RAM

# # ── Scale (fit trên benign train only) ────────────────────────────────
# scaler = StandardScaler()
# benign_mask = (y_tr == 'BenignTraffic')   # ← kiểm tra tên nhãn thực tế
# scaler.fit(X_tr[benign_mask])

# def scale_and_clean(X_arr):
#     out = scaler.transform(X_arr)
#     out[~np.isfinite(out)] = 0.0
#     return out

# # ── Export parquet ─────────────────────────────────────────────────────
# feature_cols = X_tr.columns.tolist()
# for split_name, X_split, y_split in [
#     ('train', X_tr,  y_tr),
#     ('val',   X_val, y_val),
#     ('test',  X_te,  y_te),
# ]:
#     scaled = scale_and_clean(X_split)
#     out_df = pd.DataFrame(scaled, columns=feature_cols)
#     out_df['Label'] = y_split.values
#     out_df.to_parquet(f'{OUT_DIR}{split_name}.parquet', index=False)
#     print(f"Saved {split_name}.parquet — {len(out_df):,} rows")

# with open(OUT_DIR + 'scaler_benign.pkl', 'wb') as f:
#     pickle.dump(scaler, f)
# print("Done.")

Số file: 63
Loading... (~5–10 phút, peak RAM ~14 GB)
Shape: (45019243, 40)
Columns: ['Header_Length', 'Protocol Type', 'Time_To_Live', 'Rate', 'fin_flag_number', 'syn_flag_number', 'rst_flag_number', 'psh_flag_number', 'ack_flag_number', 'ece_flag_number', 'cwr_flag_number', 'ack_count', 'syn_count', 'fin_count', 'rst_count', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP', 'SSH', 'IRC', 'TCP', 'UDP', 'DHCP', 'ARP', 'ICMP', 'IGMP', 'IPv', 'LLC', 'Tot sum', 'Min', 'Max', 'AVG', 'Std', 'Tot size', 'IAT', 'Number', 'Variance', 'Label']


In [2]:
# Chẩn đoán NaN
print(f"Total NaN in Label: {df['Label'].isna().sum():,}")
print(f"Total NaN in features: {df.drop(columns=['Label']).isna().sum().sum():,}")
print(f"Total Inf in features: {np.isinf(df.drop(columns=['Label']).select_dtypes(include=[np.number])).sum().sum():,}")
print(f"\nLabel value counts (top 10):")
print(df['Label'].value_counts(dropna=False).head(10))

Total NaN in Label: 9
Total NaN in features: 1,489
Total Inf in features: 991

Label value counts (top 10):
Label
DDOS-ICMP_FLOOD            6893259
DDOS-UDP_FLOOD             5181027
DDOS-TCP_FLOOD             4306086
DDOS-PSHACK_FLOOD          3920372
DDOS-SYN_FLOOD             3886130
DDOS-RSTFINFLOOD           3872808
DDOS-SYNONYMOUSIP_FLOOD    3445659
DOS-UDP_FLOOD              3177323
DOS-TCP_FLOOD              2558256
DOS-SYN_FLOOD              1942176
Name: count, dtype: int64


In [3]:
# Tìm label benign chính xác
print([l for l in df['Label'].unique() if 'BENIGN' in str(l).upper()])
print(f"Tổng số class: {df['Label'].nunique()}")

['BENIGN']
Tổng số class: 34


In [2]:
import numpy as np
import pandas as pd
import pickle, os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

OUT_DIR = '../data/processed/'
os.makedirs(OUT_DIR, exist_ok=True)
BENIGN_LABEL = 'BENIGN'

# ── Clean NaN/Inf ──────────────────────────────────────────────────────
n_before = len(df)
df = df.dropna(subset=['Label'])
print(f"Dropped {n_before - len(df):,} rows với Label NaN")

feature_cols = [c for c in df.columns if c != 'Label']
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
print(f"Shape sau clean: {df.shape}")

# ── Split 80/10/10 stratified ─────────────────────────────────────────
X = df.drop(columns=['Label'])
y = df['Label']

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)

print(f"Train: {len(X_tr):,} | Val: {len(X_val):,} | Test: {len(X_te):,}")
del df, X, y, X_tmp, y_tmp

# ── Scale (fit trên benign train only) ────────────────────────────────
scaler = StandardScaler()
benign_mask = (y_tr == BENIGN_LABEL)
print(f"Benign trong train: {benign_mask.sum():,}")
scaler.fit(X_tr[benign_mask])

def scale(X_arr):
    out = scaler.transform(X_arr)
    out[~np.isfinite(out)] = 0.0
    return out

# ── Export parquet ─────────────────────────────────────────────────────
for split_name, X_split, y_split in [
    ('train', X_tr,  y_tr),
    ('val',   X_val, y_val),
    ('test',  X_te,  y_te),
]:
    scaled = scale(X_split)
    out_df = pd.DataFrame(scaled, columns=feature_cols)
    out_df['Label'] = y_split.values
    out_df.to_parquet(f'{OUT_DIR}{split_name}.parquet', index=False)
    print(f"Saved {split_name}.parquet — {len(out_df):,} rows")

with open(OUT_DIR + 'scaler_benign.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Done.")

Dropped 9 rows với Label NaN
Shape sau clean: (45019234, 40)
Train: 36,015,387 | Val: 4,501,923 | Test: 4,501,924
Benign trong train: 841,098
Saved train.parquet — 36,015,387 rows
Saved val.parquet — 4,501,923 rows
Saved test.parquet — 4,501,924 rows
Done.


In [4]:
# Kiểm tra nhanh 3 parquet trước khi next step
import pandas as pd

for name in ['train', 'val', 'test']:
    df = pd.read_parquet(f'../data/processed/{name}.parquet')
    print(f"\n{name}: {df.shape}")
    print(f"  Class count: {df['Label'].nunique()}")
    print(f"  Benign: {(df['Label']=='BENIGN').sum():,}")
    print(f"  Top 3 attacks: {df['Label'].value_counts().head(3).to_dict()}")
    print(f"  NaN check: {df.isna().sum().sum()}")
    print(f"  Inf check: {df.select_dtypes(include='number').apply(lambda x: (x==float('inf')).sum()).sum()}")


train: (36015387, 40)
  Class count: 34
  Benign: 841,098
  Top 3 attacks: {'DDOS-ICMP_FLOOD': 5514607, 'DDOS-UDP_FLOOD': 4144822, 'DDOS-TCP_FLOOD': 3444869}
  NaN check: 0
  Inf check: 0

val: (4501923, 40)
  Class count: 34
  Benign: 105,137
  Top 3 attacks: {'DDOS-ICMP_FLOOD': 689326, 'DDOS-UDP_FLOOD': 518102, 'DDOS-TCP_FLOOD': 430608}
  NaN check: 0
  Inf check: 0

test: (4501924, 40)
  Class count: 34
  Benign: 105,138
  Top 3 attacks: {'DDOS-ICMP_FLOOD': 689326, 'DDOS-UDP_FLOOD': 518103, 'DDOS-TCP_FLOOD': 430609}
  NaN check: 0
  Inf check: 0


In [6]:
import pandas as pd
for name in ['train', 'val', 'test']:
    df = pd.read_parquet(f'../data/processed/{name}.parquet')
    print(f"\n{name} — 3 lớp nhỏ nhất:")
    print(df['Label'].value_counts().tail(3))


train — 3 lớp nhỏ nhất:
Label
BACKDOOR_MALWARE    2462
RECON-PINGSWEEP     1729
UPLOADING_ATTACK     957
Name: count, dtype: int64

val — 3 lớp nhỏ nhất:
Label
BACKDOOR_MALWARE    308
RECON-PINGSWEEP     216
UPLOADING_ATTACK    120
Name: count, dtype: int64

test — 3 lớp nhỏ nhất:
Label
BACKDOOR_MALWARE    308
RECON-PINGSWEEP     216
UPLOADING_ATTACK    119
Name: count, dtype: int64
